# 01 · Define & Explore — metalloenzyme design, CO₂ hydration, and the Zn-His₃-OH site

**Standard slot:** *define & explore.* **For Project 20 this means:** understand de novo
**metalloenzyme** design (the GRACE paradigm) and the carbonic-anhydrase CO₂-hydration reaction, then
**construct the metal-site theozyme** (a **Zn-His₃-OH** centre + transition-state geometry) and run a
mock theozyme→scaffold hello-world (D0).

Run `00_setup.ipynb` first in this session.

## Why carbonic anhydrase is the model CO₂-hydration metalloenzyme
Carbon capture needs fast, robust CO₂-hydration catalysts (CO₂ + H₂O ⇌ HCO₃⁻ + H⁺). **Carbonic
anhydrase (CA)** is nature's champion — a small **Zn-metalloenzyme** running near the diffusion limit.
Two properties make it the model for de novo **metal-aware** design:
- **A single, well-defined catalytic metal** — a tetrahedral **Zn(II)** held by three histidines,
  with a Zn-bound hydroxide as the nucleophile. A clean, reproducible target geometry.
- **Tractable readouts** — the classic **Wilbur-Anderson** CO₂-hydration assay, plus a promiscuous
  **esterase** activity on **p-nitrophenyl acetate (pNPA)** that gives a fast chromogenic proxy.

The honest history: **GRACE** (Hu 2024) produced functional carbonic-anhydrase-style designs — but
only by generating a **large pool (~10k)** and screening. That is the methods claim this project
tests, and the expectation it sets: *diversity before filtering, then a real assay.*

## The metal-site theozyme — the catalytic motif you must build
A **theozyme** ("theoretical enzyme") is the minimal catalytic motif placed around the **transition
state**. For a metalloenzyme that motif is the **metal centre**. For carbonic anhydrase:

| Role | Residue / species | Job in the reaction |
|------|-------------------|---------------------|
| metal ion | Zn(II) | the catalytic centre; lowers the bound-water pKa to ~7 |
| 3 × His ligands | His (imidazole N) | hold the Zn in a tetrahedral cage (**fix these in design**) |
| Zn-hydroxide | OH⁻ on Zn | the nucleophile that attacks the CO₂ carbon |
| proton shuttle | His (often a 4th) | relays the proton to bulk solvent |

You **construct** this from a verified CA structure and/or a QM transition-state model — it is a
teaching template (`data/inputs/metal_site_def.txt`), **not** fabricated experimental data. Place
groups around the **transition state**, not the resting state. This is the **enzyme-family template**
(Project 18) specialised to a **metal active site**.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Build the metal-site theozyme (mock hello-world)
`scripts/enzyme_tools.py` exposes `build_theozyme("co2_hydration")` → a Zn-His₃-OH metal-site spec.
The distances/angles it ships are **PLACEHOLDERS** — replace them in `data/inputs/metal_site_def.txt`
(and in `build_theozyme`) with real Zn-N distances / N-Zn-N angles read off a verified CA structure
(2CAB / 3KS3 — verify on RCSB) during P1.

In [ ]:
from enzyme_tools import build_theozyme

theo = build_theozyme("co2_hydration")
print("Reaction :", theo.reaction)
print("Substrate:", theo.substrate)
print("Provenance:", theo.provenance)
site = theo.metal_site
print(f"\nMetal centre: {site.metal} | coordination {site.coordination_number} "
      f"(tetrahedral) | {site.n_protein_ligands} His ligands | reactive species: {site.reactive_species}")
print("\nCatalytic functional groups (PLACEHOLDER geometry — fill from a CA structure / QM):")
for fg in theo.functional_groups:
    ang = f"{fg.target_angle}deg" if fg.target_angle is not None else "n/a"
    print(f"  {fg.role:16s} {fg.residue}/{fg.atom:4s}  d={fg.target_distance}A  angle={ang}")
print("\nMetal ligands to FIX during sequence design:", theo.catalytic_residue_ids())

## A first mock scaffold + metal-aware sequence (no GPU)
`scaffold_motif(...)` (mock) returns placeholder backbones presenting the Zn-His₃ motif;
`ligandmpnn_metal(...)` (mock) designs sequences with the **three His ligands fixed and the Zn in
context** — the central, metal-aware step. **Every number here is SYNTHETIC** — this only proves the
plumbing runs anywhere. Switch to the real backends (RFdiffusion2/Riff-Diff on an A100; metal-aware
LigandMPNN CPU-fast) in `02_generate.ipynb`.

In [ ]:
from enzyme_tools import scaffold_motif, ligandmpnn_metal, metal_ligand_geometry

scaffolds = scaffold_motif(theo, n=5, method="mock")
print(f"{len(scaffolds)} mock scaffolds; example:")
print(" ", scaffolds[0])

seqs = ligandmpnn_metal(scaffolds[0], theo.catalytic_residue_ids(), n=3, tool="mock")
print(f"\n{len(seqs)} mock METAL-AWARE sequences for {scaffolds[0]['design_id']} "
      f"(fixed roles: {seqs[0]['fixed_metal_ligand_roles']}, metal_aware={seqs[0]['metal_aware']})")
print("  fixed His positions (mock, 0-indexed):", seqs[0]['fixed_his_positions'])

geo = metal_ligand_geometry(None, theo.metal_site)   # mock, SYNTHETIC
print(f"\nmetal_ligand_geometry (mock, SYNTHETIC): rmsd={geo['metal_ligand_rmsd']}A  "
      f"Zn-N={geo['mean_zn_n_dist']}A  N-Zn-N={geo['mean_n_zn_n_angle']}deg  "
      f"coordination_ok={geo['coordination_ok']}  -> pass if rmsd < 0.5 A")
print("NOTE: these are placeholder numbers. The real campaign is in notebook 02.")

## The metrics that decide a metalloenzyme design
| Metric | Cutoff (`enzyme`) | Means | Does **not** mean |
|--------|-------------------|-------|-------------------|
| scRMSD | ≤ 2.0 Å | designed-vs-predicted backbone self-consistency | activity |
| pLDDT (global) | ≥ 85 | local fold confidence | thermostability / catalysis |
| pLDDT (catalytic) | ≥ 90 | confidence *at the His ligands* | the cage geometry is correct |
| **metal-ligand RMSD** (= `catalytic_geom_rmsd`) | **< 0.5 Å** | predicted Zn-coordinating atoms vs the target Zn-His₃ | **activity, or that the metal even binds** |

The fourth row is the point of the whole project. And the last column carries **two** messages a
metalloenzyme designer must never forget: in-silico metal geometry does not guarantee a working
enzyme, **and** good geometry does not even guarantee the **metal binds** — that is a separate,
measured check (**ICP**, notebook 05). Only an assay decides activity; only ICP decides incorporation.

> **AF2 caveat:** AF2 does **not** place the Zn. You build the metal in from the His₃ geometry (or use
> a metal-aware predictor) before scoring the metal-ligand geometry — see notebook 02/04.

## D0 checklist
- [ ] Half-page on de novo metalloenzyme design + the honest GRACE hit-rate story (large pool + screening).
- [ ] 1-page problem statement with **measurable** success criteria + the controls you'll need (apo, natural CA).
- [ ] Metal-site spec started in `data/inputs/metal_site_def.txt` (replace PLACEHOLDERs with real Zn-N distances; cite the CA structure).
- [ ] Reproduced mock hello-world (Zn-His₃-OH spec + a mock scaffold record).
- [ ] `LOG.md` entry (tool versions, GPU, seed).

**Next:** `02_generate.ipynb` — scaffold the metal motif and run **metal-aware LigandMPNN** with the three His ligands fixed.